# Drug-Drug Interaction Pipeline

A clinical decision-support research tool combining RxNorm, DDInter, PubChem, RDKit, and openFDA.

**Four functions:** `predict_interaction()`, `check_cross_reactivity()`, `check_pregnancy()`, `resolve_rxcui()`

---

## SECTION A — QUICK START (daily use)

After a kernel restart, run only these cells in order:
1. **A1** — Imports
2. **A2** — Load saved models (.pkl files)
3. **A3** — Quick Reload (rebuilds graph + defines all functions)

That's it — all four functions will be ready.

Section B (further down) contains the full build pipeline from raw data, kept for documentation.

### A1 — Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import networkx as nx
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

pd.set_option('display.max_colwidth', 100)
print('All imports OK')

All imports OK


### A2 — Load saved models (run after kernel restart)

In [2]:
import joblib, numpy as np, pandas as pd, networkx as nx

# --- load all saved objects ---
rf              = joblib.load('ddi_final_model_v2.pkl')
meta            = joblib.load('ddi_model_metadata_v2.pkl')
support         = joblib.load('ddi_support_data_v2.pkl')
feature_columns = meta['feature_columns']

ingredients       = support['ingredients']
bn_ingredient_map = support['bn_ingredient_map']
alt_names         = support['alt_names']
mol_props_full    = support['mol_props_full']
fp_matrix         = support['fp_matrix']
rxcui_to_idx      = support['rxcui_to_idx']
positive_set      = support['positive_set']
atc_lookup        = support['atc_lookup']

# severity v2 models
rf_major        = joblib.load('ddi_severity_major.pkl')
rf_minor        = joblib.load('ddi_severity_minor.pkl')
sev_config      = joblib.load('ddi_severity_config.pkl')
MAJOR_THRESHOLD = sev_config['major_threshold']
MINOR_THRESHOLD = sev_config['minor_threshold']
train_medians   = pd.Series(sev_config['train_medians'])
SEVERITY_OVERRIDES = sev_config['overrides']

ATC_PRIORITY = {'M':1,'N':2,'J':3,'C':4,'A':5,'B':6,'R':7,'G':8,'L':9,'D':10,'H':11,'P':12,'S':13,'V':14}

PROPS = ['MolecularWeight','XLogP','TPSA','HBondDonorCount',
         'HBondAcceptorCount','RotatableBondCount','HeavyAtomCount']

print('All objects loaded successfully.')
print('Model ROC-AUC was:', meta['roc_auc_test'])
print('Override list:', len(SEVERITY_OVERRIDES), 'critical pairs')


All objects loaded successfully.
Model ROC-AUC was: 0.9347
Override list: 24693 critical pairs


### ⚡ A3 — QUICK RELOAD (defines all functions)
Run this after A1 and A2. Rebuilds the interaction graph and defines every function.

In [3]:
# ============================================================
# QUICK RELOAD — run after Cell 12, after every kernel restart
# ============================================================
import networkx as nx, numpy as np, pandas as pd, json, joblib, re

# 1) rebuild interaction graph
G = nx.Graph()
G.add_edges_from(positive_set)
print(f'Graph rebuilt — nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}')
fp_rxcuis = list(rxcui_to_idx.keys())

PROPS = ['MolecularWeight','XLogP','TPSA','HBondDonorCount',
         'HBondAcceptorCount','RotatableBondCount','HeavyAtomCount']
mol_feat_cols = [f'{fn}_{p}' for fn in ['diff','sum'] for p in PROPS]

# 2) load severity v2 models
rf_major           = joblib.load('ddi_severity_major.pkl')
rf_minor           = joblib.load('ddi_severity_minor.pkl')
sev_config         = joblib.load('ddi_severity_config.pkl')
MAJOR_THRESHOLD    = sev_config['major_threshold']
MINOR_THRESHOLD    = sev_config['minor_threshold']
train_medians      = pd.Series(sev_config['train_medians'])
severity_feature_columns = sev_config.get('severity_feature_columns', feature_columns)
new_features             = sev_config.get('new_features', [])
SEVERITY_OVERRIDES = sev_config['overrides']
UNKNOWN_SEVERITY_PAIRS = sev_config.get('unknown_severity_pairs', set())
print(f'Severity models loaded. Overrides: {len(SEVERITY_OVERRIDES):,} pairs')

# 3) load allergy config
allergy_config     = joblib.load('ddi_allergy_config.pkl')
TANIMOTO_THRESHOLD = allergy_config['tanimoto_threshold']
ATC_MIN_TANIMOTO   = allergy_config['atc_min_tanimoto']
PHARMACOPHORE_CLASSES = {k: set(v) for k, v in allergy_config['pharmacophore_classes'].items()}
support_data     = joblib.load('ddi_support_data_v2.pkl')
REGIONAL_SYNONYMS = support_data.get('regional_synonyms', {})
PRIORITY_OVERRIDES = support_data.get('priority_overrides', {})
rxcui_to_atc      = support_data.get('rxcui_to_atc', {})
HIGH_RISK_ATC     = support_data.get('high_risk_atc', {})
print(f'Regional synonyms loaded: {len(REGIONAL_SYNONYMS)}')
print(f'Allergy config loaded. Pharmacophore classes: {len(PHARMACOPHORE_CLASSES)}')

# 4) resolve_rxcui
def resolve_rxcui(name):
    name_lower = name.lower().strip()
    if name_lower in PRIORITY_OVERRIDES:
        return PRIORITY_OVERRIDES[name_lower], 'priority_override'
    hit = ingredients[ingredients['name_lower'] == name_lower]
    if len(hit) > 0: return hit.iloc[0]['RXCUI'], 'ingredient'
    bn = bn_ingredient_map.copy()
    bn['name_lower'] = bn['brand_name'].str.lower().str.strip()
    hit = bn[bn['name_lower'] == name_lower]
    if len(hit) > 0:
        ing = ingredients[ingredients['ingredient_name'] == hit.iloc[0]['ingredient_name']]
        if len(ing) > 0: return ing.iloc[0]['RXCUI'], 'brand_name'
    hit = alt_names[alt_names['name_lower'] == name_lower]
    if len(hit) > 0: return hit.iloc[0]['RXCUI'], 'synonym'
    if name_lower in REGIONAL_SYNONYMS:
        return REGIONAL_SYNONYMS[name_lower], 'regional_brand'
    return None, None
def get_primary_atc_class(rxcui):
    classes = atc_lookup[atc_lookup['RXCUI'] == rxcui]['atc_class'].unique()
    if len(classes) == 0: return None
    sizes = {c: atc_lookup[atc_lookup['atc_class']==c]['RXCUI'].nunique() for c in classes}
    return min(classes, key=lambda c: (ATC_PRIORITY.get(c[0], 99), -sizes[c]))

def get_alternatives(name_a, name_b, max_alternatives=5):
    rxcui_a, _ = resolve_rxcui(name_a)
    rxcui_b, _ = resolve_rxcui(name_b)
    if rxcui_a is None or rxcui_b is None: return {'error': 'Could not resolve one or both drug names.'}
    primary_class = get_primary_atc_class(rxcui_b)
    if primary_class is None: return {'message': f'No ATC class found for {name_b}.'}
    same_class = [r for r in atc_lookup[atc_lookup['atc_class']==primary_class]['RXCUI'].unique()
                  if r != rxcui_b and r != rxcui_a]
    interacting_with_a = {(p[0] if p[1]==rxcui_a else p[1]) for p in positive_set if rxcui_a in p}
    safe = sorted([r for r in same_class if r not in interacting_with_a],
                  key=lambda r: G.degree(r) if G.has_node(r) else 0, reverse=True)
    rxcui_to_name = ingredients.set_index('RXCUI')['ingredient_name'].to_dict()
    candidates = [{'name': rxcui_to_name[r], 'interaction_degree': G.degree(r) if G.has_node(r) else 0}
                  for r in safe if r in rxcui_to_name][:max_alternatives]
    return {'drug_a': name_a, 'drug_b': name_b, 'atc_class': primary_class, 'alternatives': candidates,
            'note': (f'Alternatives share ATC class {primary_class} with {name_b} '
                     f'and have no recorded interaction with {name_a} in DDInter. '
                     f'Always verify with a pharmacist.')}

# 6) predict_severity
def get_drug_risk_score(rxcui):
    rxcui = str(rxcui)
    atc_letter = rxcui_to_atc.get(rxcui, '')
    atc_score  = HIGH_RISK_ATC.get(atc_letter, 0)
    pharm_score = 0
    for cls, members in PHARMACOPHORE_CLASSES.items():
        if rxcui in members:
            pharm_score = 2 if cls in ['nsaid','opioid','statin'] else 1
            break
    return atc_score, pharm_score

def build_severity_features_from_base(feat, rxcui_a, rxcui_b):
    atc_a, pharm_a = get_drug_risk_score(rxcui_a)
    atc_b, pharm_b = get_drug_risk_score(rxcui_b)
    feat = dict(feat)
    feat['atc_risk_sum']   = atc_a + atc_b
    feat['atc_risk_max']   = max(atc_a, atc_b)
    feat['atc_risk_prod']  = atc_a * atc_b
    feat['pharm_risk_sum'] = pharm_a + pharm_b
    feat['pharm_risk_max'] = max(pharm_a, pharm_b)
    feat['both_high_risk'] = 1 if (atc_a >= 2 and atc_b >= 2) else 0
    atc_la = rxcui_to_atc.get(str(rxcui_a), '')
    atc_lb = rxcui_to_atc.get(str(rxcui_b), '')
    feat['same_atc_class']    = 1 if (atc_la and atc_la == atc_lb) else 0
    feat['blood_interaction'] = 1 if ('B' in [atc_la, atc_lb]) else 0
    return feat

def predict_severity(X, rxcui_a, rxcui_b):
    key = (min(rxcui_a, rxcui_b), max(rxcui_a, rxcui_b))
    if key in SEVERITY_OVERRIDES:
        return SEVERITY_OVERRIDES[key], 'OVERRIDE', {
            'major_probability': None, 'minor_probability': None,
            'note': 'Severity from verified clinical override list'}
    base_feat = X.iloc[0].to_dict()
    enh_feat  = build_severity_features_from_base(base_feat, rxcui_a, rxcui_b)
    X_enh     = pd.DataFrame([enh_feat])[severity_feature_columns].fillna(train_medians)
    str_key = (min(str(rxcui_a), str(rxcui_b)), max(str(rxcui_a), str(rxcui_b)))
    in_unknown = str_key in UNKNOWN_SEVERITY_PAIRS
    major_p = rf_major.predict_proba(X_enh)[0, 1]
    minor_p = rf_minor.predict_proba(X_enh)[0, 1]
    if major_p >= MAJOR_THRESHOLD: severity = 'Major'; top_prob = major_p
    elif minor_p >= MINOR_THRESHOLD: severity = 'Minor'; top_prob = minor_p
    else: severity = 'Moderate'; top_prob = 1 - major_p
    if in_unknown:
        return severity, 'UNCERTAIN', {
            'major_probability': round(float(major_p), 3),
            'minor_probability': round(float(minor_p), 3),
            'note': 'DDInter records this interaction but does NOT classify its severity. Model estimate only — verify clinically.'}
    sev_conf = ('HIGH' if top_prob >= 0.70 else 'MEDIUM' if top_prob >= 0.55 else 'LOW')
    return severity, sev_conf, {'major_probability': round(float(major_p),3), 'minor_probability': round(float(minor_p),3)}
def predict_interaction(name_a, name_b, include_alternatives=True, max_alternatives=5):
    rxcui_a, src_a = resolve_rxcui(name_a)
    rxcui_b, src_b = resolve_rxcui(name_b)
    if rxcui_a is None: return {'error': f"Could not find '{name_a}'"}
    if rxcui_b is None: return {'error': f"Could not find '{name_b}'"}
    if rxcui_a == rxcui_b: return {'error': 'Both names resolve to the same ingredient.'}
    a, b = rxcui_a, rxcui_b
    override_key = (min(a,b), max(a,b))
    in_override  = override_key in SEVERITY_OVERRIDES
    in_a, in_b   = G.has_node(a), G.has_node(b)
    da = G.degree(a) if in_a else 0; db = G.degree(b) if in_b else 0
    if in_a and in_b:
        na, nb  = set(G.neighbors(a)), set(G.neighbors(b))
        common  = len(na & nb); union = len(na | nb)
        jaccard = common/union if union else 0
        adamic  = sum(1/np.log(G.degree(n)) for n in (na & nb) if G.degree(n) > 1)
    else: common = jaccard = adamic = 0
    m = mol_props_full.set_index('RXCUI')
    feat = {'deg_a':da,'deg_b':db,'common_neighbors':common,'jaccard':jaccard,'adamic_adar':adamic}
    for p in PROPS:
        va = m.loc[a,p] if a in m.index else m[p].median()
        vb = m.loc[b,p] if b in m.index else m[p].median()
        feat[f'diff_{p}'] = abs(va-vb); feat[f'sum_{p}'] = va+vb
    if a in rxcui_to_idx and b in rxcui_to_idx:
        va, vb = fp_matrix[rxcui_to_idx[a]], fp_matrix[rxcui_to_idx[b]]
        inter  = np.logical_and(va,vb).sum(); uni = np.logical_or(va,vb).sum()
        feat['tanimoto'] = inter/uni if uni else 0; feat['has_fingerprint'] = 1
    else: feat['tanimoto'] = 0; feat['has_fingerprint'] = 0
    X = pd.DataFrame([feat])[feature_columns]
    if in_override: proba = 1.0; interacts = True
    else: proba = rf.predict_proba(X)[0,1]; interacts = proba >= 0.5
    if interacts and in_a and in_b: severity, sev_conf, sev_probs = predict_severity(X, a, b)
    elif interacts and in_override:
        severity = SEVERITY_OVERRIDES[override_key]; sev_conf = 'OVERRIDE'
        sev_probs = {'major_probability': None, 'minor_probability': None, 'note': 'Severity from verified clinical override list'}
    else: severity = sev_conf = sev_probs = None
    if interacts and include_alternatives:
        alt = get_alternatives(name_a, name_b, max_alternatives)
        alts = alt.get('alternatives', []); atc_c = alt.get('atc_class', None)
        alt_note = f'Same ATC class as {name_b}, no recorded interaction with {name_a}. Always verify with a pharmacist.'
    else: alts = []; atc_c = None; alt_note = 'No interaction detected — no alternatives needed.'
    if not in_a or not in_b:
        missing = ([name_a] if not in_a else []) + ([name_b] if not in_b else [])
        confidence = 'NORMAL' if in_override else 'LOW'
        warning = None if in_override else f"{' and '.join(missing)} have no recorded interactions in DDInter. This prediction is NOT reliable for clinical use."
    else: confidence = 'NORMAL'; warning = None
    result = {
        'drug_a': name_a, 'resolved_as_a': src_a,
        'drug_b': name_b, 'resolved_as_b': src_b,
        'interaction_probability': round(float(proba),4),
        'prediction': 'Interaction likely' if interacts else 'No interaction expected',
        'severity': severity, 'severity_confidence': sev_conf,
        'severity_confidence_explanation': {
            'OVERRIDE': 'Severity verified from DDInter clinical data — highest reliability',
            'HIGH': 'Model is >70% confident in this severity level',
            'MEDIUM': 'Model is 55-70% confident — use with caution',
            'LOW': 'Model confidence is below 55% — severity is uncertain',
            'UNCERTAIN': 'Model predicts Moderate but Major probability is borderline (0.25-0.35) — verify clinically',
        }.get(sev_conf, None),
        'severity_probabilities': sev_probs, 'confidence': confidence,
        'known_in_training_data': (a,b) in positive_set or (b,a) in positive_set,
        'alternatives': {'atc_class': atc_c, 'candidates': alts, 'note': alt_note}
    }
    if warning: result['warning'] = warning
    return result

# 8) allergy helpers
SALT_PATTERN = re.compile(
    r'\s+(sodium|potassium|calcium|magnesium|hydrochloride|sulfate|phosphate|'
    r'anhydrous|monohydrate|dihydrate|trihydrate|hemihydrate|pentahydrate|'
    r'acetate|citrate|tartrate|maleate|fumarate|succinate|besylate|mesylate|'
    r'tosylate|tromethamine|meglumine|benzathine|procaine|pivoxil|axetil|'
    r'proxetil|disodium|dipotassium|polistirex|tannate|camsyl|lactate|'
    r'napsylate|hydrate|anhyd|bitartrate|trometamol|terephthalate).*$',
    re.IGNORECASE
)

def get_base_name(drug_name):
    name = drug_name.lower().strip()
    name = name.split(',')[0].strip()
    name = SALT_PATTERN.sub('', name).strip()
    name = re.sub(r'\s*\(.*?\)', '', name).strip()
    return name

def get_pharmacophore_class(rxcui):
    return [cls for cls, rxcui_set in PHARMACOPHORE_CLASSES.items() if rxcui in rxcui_set]

def check_cross_reactivity(allergic_to, max_results=10,
                            tanimoto_threshold=None, atc_min_tanimoto=None):
    tanimoto_threshold = tanimoto_threshold or TANIMOTO_THRESHOLD
    atc_min_tanimoto   = atc_min_tanimoto   or ATC_MIN_TANIMOTO
    rxcui_a, src = resolve_rxcui(allergic_to)
    if rxcui_a is None: return {'error': f"Could not find '{allergic_to}' in database."}
    primary_class   = get_primary_atc_class(rxcui_a)
    pharm_classes   = get_pharmacophore_class(rxcui_a)
    same_atc_rxcuis = set(atc_lookup[atc_lookup['atc_class']==primary_class]['RXCUI'].unique()) if primary_class else set()
    same_pharm_rxcuis = set()
    for cls in pharm_classes: same_pharm_rxcuis.update(PHARMACOPHORE_CLASSES[cls])
    same_pharm_rxcuis.discard(rxcui_a)
    pure_ingredients = set(ingredients[ingredients['TTY'].isin(['IN','PIN'])]['RXCUI'].tolist())
    allergic_base = get_base_name(allergic_to); results = {}; seen_bases = {allergic_base}
    def try_add(rxcui_b, detected_by, tanimoto=None, risk='HIGH'):
        if rxcui_b in results: return
        if rxcui_b not in pure_ingredients: return
        name_vals = ingredients[ingredients['RXCUI']==rxcui_b]['ingredient_name'].values
        if len(name_vals) == 0: return
        base = get_base_name(name_vals[0])
        if base in seen_bases: return
        seen_bases.add(base)
        results[rxcui_b] = {'name': base, 'tanimoto': round(tanimoto,4) if tanimoto is not None else None,
                             'detected_by': detected_by, 'risk': risk}
    if rxcui_a in rxcui_to_idx:
        va = fp_matrix[rxcui_to_idx[rxcui_a]]
        inter = np.logical_and(fp_matrix,va).sum(axis=1); union = np.logical_or(fp_matrix,va).sum(axis=1)
        tan_scores = np.where(union>0, inter/union, 0.0)
        for i, score in enumerate(tan_scores):
            rxcui_b = fp_rxcuis[i]
            if rxcui_b == rxcui_a or score < tanimoto_threshold: continue
            if rxcui_b not in same_atc_rxcuis and rxcui_b not in same_pharm_rxcuis and score < 0.50: continue
            try_add(rxcui_b, 'structural_similarity', score, 'HIGH' if score>=0.40 else 'MODERATE')
    for rxcui_b in same_pharm_rxcuis:
        if rxcui_a in rxcui_to_idx and rxcui_b in rxcui_to_idx:
            va=fp_matrix[rxcui_to_idx[rxcui_a]]; vb=fp_matrix[rxcui_to_idx[rxcui_b]]
            inter=np.logical_and(va,vb).sum(); union=np.logical_or(va,vb).sum()
            t=float(inter/union) if union>0 else 0.0
        else: t=None
        if t is not None and t < atc_min_tanimoto: continue
        try_add(rxcui_b, 'pharmacophore_class', t, 'HIGH')
    for rxcui_b in same_atc_rxcuis:
        if rxcui_b == rxcui_a: continue
        if rxcui_a in rxcui_to_idx and rxcui_b in rxcui_to_idx:
            va=fp_matrix[rxcui_to_idx[rxcui_a]]; vb=fp_matrix[rxcui_to_idx[rxcui_b]]
            inter=np.logical_and(va,vb).sum(); union=np.logical_or(va,vb).sum()
            t=float(inter/union) if union>0 else 0.0
        else: t=None
        if t is None or t < atc_min_tanimoto: continue
        try_add(rxcui_b, 'atc_class', t, 'MODERATE')
    if not results:
        return {'allergic_to': allergic_to, 'message': 'No cross-reactive drugs found.',
                'atc_class': primary_class, 'pharmacophore_class': pharm_classes if pharm_classes else None}
    sorted_results = sorted(results.values(),
        key=lambda x: (0 if x['detected_by']=='pharmacophore_class' else
                        1 if x['detected_by']=='structural_similarity' else 2,
                        0 if x['tanimoto'] is not None else 1, -(x['tanimoto'] or 0)))[:max_results]
    has_high = any(d['risk']=='HIGH' for d in sorted_results)
    uncertainty_note = None if (pharm_classes or has_high) else (
        'No known pharmacophore cross-reactivity class for this drug. '
        'Results are based on structural similarity only and may NOT '
        'represent true clinical cross-reactivity risk. '
        'Consult a clinical pharmacist for guidance.')
    return {'allergic_to': allergic_to, 'resolved_as': src, 'atc_class': primary_class,
            'pharmacophore_class': pharm_classes if pharm_classes else None,
            'threshold_used': tanimoto_threshold, 'cross_reactive_drugs': sorted_results,
            'uncertainty_note': uncertainty_note,
            'note': ('These drugs share structural similarity or pharmacological class '
                     'with the drug you are allergic to. Cross-reactivity risk varies. '
                     'Always consult a doctor or pharmacist before use.')}

# load pregnancy database
# load pregnancy database
preg_data           = joblib.load('ddi_pregnancy_data.pkl')
PREGNANCY_OVERRIDES = preg_data['overrides']
pregnancy_db        = preg_data['pregnancy_db']
PREGNANCY_RULES = [
    (r'contraindicated.{0,80}pregnan|category\s*x', 'X', 'Contraindicated in pregnancy — risk of fetal harm'),
    (r'teratogen|embryotoxic|fetal harm|fetal risk|neonatal.{0,50}death', 'D', 'Evidence of fetal risk — use only if benefits outweigh risks'),
    (r'20 weeks|after 20|third trimester|last 3 months|oligohydramnios', 'D*', 'Avoid in late pregnancy (after 20 weeks)'),
    (r'no adequate.{0,50}well.controlled|cross.{0,30}placenta|category\s*c|benefit.{0,50}outweigh', 'C', 'Use with caution — no adequate human studies'),
    (r'category\s*b|no evidence.{0,50}harm|not reported a clear', 'B', 'Relatively safe — no evidence of harm in animal studies'),
    (r'not sufficient to inform|insufficient data|limited.{0,50}data', 'N/A', 'Insufficient data — consult physician'),
    (r'category\s*a', 'A', 'Safe — controlled human studies show no risk'),
]
def fetch_pregnancy_openfda(drug_name):
    import requests
    resp = requests.get('https://api.fda.gov/drug/label.json',
                        params={'search': f'openfda.generic_name:"{drug_name}"','limit':1}, timeout=10)
    if resp.status_code != 200: return {'error': 'API failed'}
    data = resp.json()
    if 'results' not in data: return {'error': 'No label found'}
    label = data['results'][0]
    fields = ['pregnancy','pregnancy_or_breast_feeding','teratogenic_effects','use_in_specific_populations']
    return {'data': {f: ' '.join(label[f]) if isinstance(label.get(f,[]),list) else str(label.get(f,'')) for f in fields if f in label}}
def extract_pregnancy_category(text, drug_name=None):
    if drug_name and drug_name.lower() in PREGNANCY_OVERRIDES:
        cat, warn = PREGNANCY_OVERRIDES[drug_name.lower()]; return cat, warn, 'clinical_override'
    text_lower = text.lower()
    for pattern, category, warning in PREGNANCY_RULES:
        if re.search(pattern, text_lower): return category, warning, 'text_extraction'
    return 'Unknown', 'No pregnancy data found — consult physician', 'not_found'
def get_pregnancy_info(drug_name):
    rxcui, _ = resolve_rxcui(drug_name)
    ingredient = drug_name
    if rxcui:
        ing = ingredients[ingredients['RXCUI']==rxcui]['ingredient_name'].values
        if len(ing) > 0: ingredient = ing[0]
    pllr_note = None
    if ingredient in pregnancy_db:
        data = pregnancy_db[ingredient]; cat=data['category']; warn=data['warning']; src=data['source']
        if src != 'clinical_override':
            raw=fetch_pregnancy_openfda(ingredient); all_text=' '.join(raw.get('data',{}).values()) if 'data' in raw else ''
            m=re.search(r'risk summary\s+(.{50,250}?)(?:\.|in animal|because|treatment)',all_text,re.IGNORECASE)
            if m: pllr_note=m.group(1).strip()
        return {'drug':drug_name,'ingredient':ingredient,'category':cat,'warning':warn,'source':src,'pllr_note':pllr_note}
    raw=fetch_pregnancy_openfda(ingredient); all_text=' '.join(raw.get('data',{}).values()) if 'data' in raw else ''
    cat,warn,src=extract_pregnancy_category(all_text,ingredient)
    m=re.search(r'risk summary\s+(.{50,250}?)(?:\.|in animal|because|treatment)',all_text,re.IGNORECASE)
    if m: pllr_note=m.group(1).strip()
    return {'drug':drug_name,'ingredient':ingredient,'category':cat,'warning':warn,'source':src,'pllr_note':pllr_note}
def check_pregnancy(name_a, name_b=None):
    ra=get_pregnancy_info(name_a)
    if name_b is None:
        return {'drug':name_a,'ingredient':ra['ingredient'],'category':ra['category'],
                'warning':ra['warning'],'pllr_note':ra.get('pllr_note'),'source':ra['source'],
                'note':'Always consult a physician before taking any medication during pregnancy.'}
    rb=get_pregnancy_info(name_b)
    rr={'X':5,'D':4,'D*':3,'C':2,'B':1,'A':0,'N/A':1,'Unknown':1}
    hi=ra['category'] if rr.get(ra['category'],1)>=rr.get(rb['category'],1) else rb['category']
    adv={'X':'⛔ AVOID — one or both drugs are contraindicated in pregnancy',
         'D':'🔴 HIGH RISK — use only if no safe alternative exists',
         'D*':'🟠 AVOID IN LATE PREGNANCY — especially after 20 weeks',
         'C':'🟡 USE WITH CAUTION — consult physician before use',
         'B':'🟢 RELATIVELY SAFE — but always confirm with physician',
         'A':'✅ SAFE — generally considered safe in pregnancy'}.get(hi,'⚪ UNKNOWN — consult physician')
    return {'drug_a':{'name':name_a,'ingredient':ra['ingredient'],'category':ra['category'],'warning':ra['warning'],'pllr_note':ra.get('pllr_note')},
            'drug_b':{'name':name_b,'ingredient':rb['ingredient'],'category':rb['category'],'warning':rb['warning'],'pllr_note':rb.get('pllr_note')},
            'overall_risk':hi,'overall_advice':adv,
            'note':'Always consult a physician before taking any medication during pregnancy.'}
print(f'Pregnancy DB loaded: {len(pregnancy_db)} drugs')
print('All functions ready. Pipeline loaded successfully.')
print('Interaction test:', predict_interaction('Warfarin','Aspirin')['severity'],
      '—', predict_interaction('Warfarin','Aspirin')['severity_confidence'])
print('Lithium test    :', predict_interaction('Lithium','Ibuprofen')['prediction'],
      '—', predict_interaction('Lithium','Ibuprofen')['severity'])
print('Allergy test    :', len(check_cross_reactivity('penicillin G')['cross_reactive_drugs']),
      'cross-reactive drugs for penicillin G')
print('Panadol test    :', check_cross_reactivity('panadol')['pharmacophore_class'])
print('Regional test   :', resolve_rxcui('Nurofen')[1], resolve_rxcui('Concor')[1])


Graph rebuilt — nodes: 1706, edges: 143204
Severity models loaded. Overrides: 24,693 pairs
Regional synonyms loaded: 59
Allergy config loaded. Pharmacophore classes: 13
Pregnancy DB loaded: 1712 drugs
All functions ready. Pipeline loaded successfully.
Interaction test: Major — OVERRIDE
Lithium test    : Interaction likely — Major
Allergy test    : 10 cross-reactive drugs for penicillin G
Panadol test    : ['para_aminophenol']
Regional test   : regional_brand regional_brand


# Testing :  التفاعل + الخطورة + البدائل + الحمل + الحساسية 

In [5]:
import json

# ============================================================
# COMPREHENSIVE TEST CELL — all four functions
# ============================================================

def test_full(drug_a, drug_b):
    """Run interaction + severity + allergy + pregnancy for a drug pair."""
    print("=" * 70)
    print(f"  {drug_a.upper()}  +  {drug_b.upper()}")
    print("=" * 70)

    # 1) INTERACTION + SEVERITY
    r = predict_interaction(drug_a, drug_b)
    if "error" in r:
        print(f"  ❌ {r['error']}")
        return
    print(f"\n  🔹 INTERACTION")
    print(f"     Prediction:  {r['prediction']}")
    print(f"     Probability: {r['interaction_probability']}")
    print(f"     Severity:    {r.get('severity')} ({r.get('severity_confidence')})")
    print(f"     Confidence:  {r.get('confidence')}")
    if r.get("warning"):
        print(f"     ⚠️  {r['warning'][:60]}")

    # alternatives
    alts = r.get("alternatives", {}).get("candidates", [])
    if alts:
        names = ", ".join(a["name"] for a in alts[:3])
        print(f"     Alternatives: {names}")

    # 2) PREGNANCY (pair)
    p = check_pregnancy(drug_a, drug_b)
    print(f"\n  🔹 PREGNANCY")
    print(f"     {drug_a}: {p['drug_a']['category']} — {p['drug_a']['warning'][:45]}")
    print(f"     {drug_b}: {p['drug_b']['category']} — {p['drug_b']['warning'][:45]}")
    print(f"     Overall: {p['overall_risk']} | {p['overall_advice']}")

    # 3) ALLERGY (each drug separately)
    print(f"\n  🔹 ALLERGY CROSS-REACTIVITY")
    for drug in [drug_a, drug_b]:
        a = check_cross_reactivity(drug)
        cls = a.get("pharmacophore_class") or ["None"]
        cross = a.get("cross_reactive_drugs", [])
        top = ", ".join(d["name"] for d in cross[:3]) if cross else "none found"
        print(f"     {drug:12} class: {str(cls):28} → {top}")
    print()


# ============================================================
# RUN TESTS
# ============================================================
test_pairs = [
    ("Warfarin",    "Aspirin"),      # Major OVERRIDE, both risky in pregnancy
    ("Lipitor",     "Klacid"),       # Major OVERRIDE (statin + macrolide)
    ("Nurofen",     "Coumadin"),     # regional brands → Major
    ("Panadol",     "Amoxil"),       # safe combo
    ("Epilim",      "Klacid"),       # valproate X in pregnancy
    ("Glucophage",  "Concor"),       # diabetes + beta-blocker
    ("dabigatran",  "clarithromycin"), # new drug Major OVERRIDE
    ("Bactrim",     "Warfarin"),     # sulfonamide allergy + interaction
]

for a, b in test_pairs:
    test_full(a, b)

  WARFARIN  +  ASPIRIN

  🔹 INTERACTION
     Prediction:  Interaction likely
     Probability: 1.0
     Severity:    Major (OVERRIDE)
     Confidence:  NORMAL
     Alternatives: opium, oliceridine, lasmiditan

  🔹 PREGNANCY
     Warfarin: X — Contraindicated in pregnancy — risk of fetal 
     Aspirin: D* — Avoid in late pregnancy (after 20 weeks) — ri
     Overall: X | ⛔ AVOID — one or both drugs are contraindicated in pregnancy

  🔹 ALLERGY CROSS-REACTIVITY
     Warfarin     class: ['None']                     → acenocoumarol, dicumarol, clopidogrel
     Aspirin      class: ['nsaid']                    → ketoprofen lysine, phenylbutazone, piketoprofen

  LIPITOR  +  KLACID

  🔹 INTERACTION
     Prediction:  Interaction likely
     Probability: 1.0
     Severity:    Major (OVERRIDE)
     Confidence:  NORMAL
     Alternatives: levofloxacin, ciprofloxacin, ofloxacin

  🔹 PREGNANCY
     Lipitor: X — Statins contraindicated in pregnancy
     Klacid: C — Use with caution — animal studies sh

### A4 — Model Evaluation Summary (all metrics + citations)

In [4]:
# ============================================================
# COMPLETE MODEL EVALUATION SUMMARY
# ============================================================
# This cell documents all measured metrics for the pipeline.
# Metrics were computed on held-out test sets and clinical ground truth.

evaluation_summary = {
    'interaction_detection': {
        'roc_auc': 0.945, 'accuracy': 0.871, 'precision': 0.854,
        'recall': 0.895, 'f1': 0.874,
        'test_set': '52,895 pairs (DDInter, 50/50 balanced)',
        'method': 'RandomForest, 21 features (graph + molecular + fingerprint)'
    },
    'severity_major': {
        'roc_auc': 0.903, 'recall': 0.86, 'precision': 0.54, 'f1': 0.66,
        'test_set': '22,679 pairs (DDInter labeled severity)',
        'method': 'RandomForest, 29 features (added 8 pharmacology/ATC features)'
    },
    'severity_minor': {
        'roc_auc': 0.938, 'recall': 0.83, 'precision': 0.34, 'f1': 0.48,
        'test_set': '22,679 pairs (DDInter labeled severity)'
    },
    'allergy_cross_reactivity': {
        'accuracy': 0.918, 'precision': 0.904, 'recall': 0.945, 'f1': 0.924,
        'test_set': '208 pairs (clinical ground truth, 10 drug classes)',
        'method': 'Tanimoto + 13 pharmacophore classes + ATC fallback',
        'benchmark': 'Beta-lactam clinical models: 51-60% sensitivity (ScienceDirect 2017)'
    },
    'pregnancy_extraction': {
        'accuracy_text_extraction': 0.75,
        'accuracy_overall': 0.826,
        'test_set': '46 drugs (clinical ground truth)',
        'benchmark': 'Rodriguez & Demner-Fushman 2015 (PMC4765680): 0.79 accuracy',
        'method': 'openFDA label text + regex extraction + PLLR notes'
    },
}

import json
print(json.dumps(evaluation_summary, indent=2))

print('\n' + '='*65)
print('CITATIONS FOR THESIS')
print('='*65)
citations = [
    'DDInter: Xian et al. (2022) Nucleic Acids Research',
    'RxNorm: Nelson et al. (2011) NLM/NIH',
    'PubChem: Kim et al. (2023) Nucleic Acids Research',
    'RDKit: open-source cheminformatics (rdkit.org)',
    'openFDA: U.S. FDA drug label API',
    'Tanimoto similarity: Bajusz et al. (2015) J Cheminformatics',
    'Pregnancy ML benchmark: Rodriguez & Demner-Fushman (2015) AMIA, PMC4765680',
    'Beta-lactam allergy models: ScienceDirect (2017) S2213219817303653',
    'Tanimoto limitations: Frontiers Bioinformatics (2025)',
]
for c in citations:
    print(f'  - {c}')


{
  "interaction_detection": {
    "roc_auc": 0.945,
    "accuracy": 0.871,
    "precision": 0.854,
    "recall": 0.895,
    "f1": 0.874,
    "test_set": "52,895 pairs (DDInter, 50/50 balanced)",
    "method": "RandomForest, 21 features (graph + molecular + fingerprint)"
  },
  "severity_major": {
    "roc_auc": 0.903,
    "recall": 0.86,
    "precision": 0.54,
    "f1": 0.66,
    "test_set": "22,679 pairs (DDInter labeled severity)",
    "method": "RandomForest, 29 features (added 8 pharmacology/ATC features)"
  },
  "severity_minor": {
    "roc_auc": 0.938,
    "recall": 0.83,
    "precision": 0.34,
    "f1": 0.48,
    "test_set": "22,679 pairs (DDInter labeled severity)"
  },
  "allergy_cross_reactivity": {
    "accuracy": 0.918,
    "precision": 0.904,
    "recall": 0.945,
    "f1": 0.924,
    "test_set": "208 pairs (clinical ground truth, 10 drug classes)",
    "method": "Tanimoto + 13 pharmacophore classes + ATC fallback",
    "benchmark": "Beta-lactam clinical models: 51-60% sen

---

## SECTION B — FULL BUILD PIPELINE (archive)

These cells build the entire system from raw data (RxNorm RRF files + DDInter downloads + PubChem API).
**You do NOT need to run these for daily use** — Section A loads the saved results.

Run these only to rebuild from scratch. Note: Cell B6 (PubChem fetch) takes ~15 minutes.

Order: B1 → B2 → ... → B11 builds the interaction model. B14+ build severity, allergy, and pregnancy.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import networkx as nx
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

pd.set_option('display.max_colwidth', 100)
print('All imports OK')

All imports OK


## Cell 2 — Load RXNCONSO + RXNREL

In [ ]:
CONSO_COLS = ['RXCUI','LAT','TS','LUI','STT','SUI','ISPREF','RXAUI','SAUI',
              'SCUI','SDUI','SAB','TTY','CODE','STR','SRL','SUPPRESS','CVF']
REL_COLS   = ['RXCUI1','RXAUI1','STYPE1','REL','RXCUI2','RXAUI2','STYPE2',
              'RELA','RUI','SRUI','SAB','SL','RG','DIR','SUPPRESS','CVF']

rxnconso = pd.read_csv(
    'RXNCONSO.RRF', sep='|', header=None,
    names=CONSO_COLS + ['EXTRA'], dtype=str, keep_default_na=False
).drop(columns=['EXTRA'])

rxnrel = pd.read_csv(
    'RXNREL.RRF', sep='|', header=None,
    names=REL_COLS + ['EXTRA'], dtype=str, keep_default_na=False
).drop(columns=['EXTRA'])

rxnorm_only   = rxnconso[rxnconso['SAB'] == 'RXNORM']
rxnrel_rxnorm = rxnrel[rxnrel['SAB'] == 'RXNORM']

print('RXNCONSO:', rxnconso.shape)
print('RXNREL:  ', rxnrel.shape)

## Cell 3 — Build drug_ingredient_map

In [ ]:
ingredients = (
    rxnorm_only[rxnorm_only['TTY'].isin(['IN','PIN','MIN'])]
    [['RXCUI','TTY','STR']].drop_duplicates()
    .rename(columns={'STR':'ingredient_name'})
)
ingredients['name_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

drugs = (
    rxnorm_only[rxnorm_only['TTY'].isin(['SCD','SBD','GPCK','BPCK'])]
    [['RXCUI','TTY','STR']].drop_duplicates()
    .rename(columns={'STR':'drug_name'})
)

# two-hop join: SCD/SBD -[consists_of]-> SCDC -[has_ingredient]-> IN
has_ing     = rxnrel_rxnorm[rxnrel_rxnorm['RELA'] == 'has_ingredient']
consists_of = rxnrel_rxnorm[rxnrel_rxnorm['RELA'] == 'consists_of']
contains    = rxnrel_rxnorm[rxnrel_rxnorm['RELA'] == 'contains']

bridge = consists_of[['RXCUI1','RXCUI2']].rename(
    columns={'RXCUI1':'component_rxcui','RXCUI2':'drug_rxcui'})
ing_link = has_ing[['RXCUI1','RXCUI2']].rename(
    columns={'RXCUI1':'ingredient_rxcui','RXCUI2':'component_rxcui'})
path = bridge.merge(ing_link, on='component_rxcui')

final = path.merge(drugs, left_on='drug_rxcui', right_on='RXCUI')
final = final.merge(ingredients, left_on='ingredient_rxcui', right_on='RXCUI', suffixes=('_drug','_ing'))

pack_bridge = contains[['RXCUI1','RXCUI2']].rename(
    columns={'RXCUI1':'drug_rxcui','RXCUI2':'pack_rxcui'})
pack_path  = pack_bridge.merge(path[['drug_rxcui','ingredient_rxcui']], on='drug_rxcui')
pack_final = pack_path.merge(drugs, left_on='pack_rxcui', right_on='RXCUI')
pack_final = pack_final.merge(ingredients, left_on='ingredient_rxcui', right_on='RXCUI', suffixes=('_drug','_ing'))

cols = ['RXCUI_drug','TTY_drug','drug_name','RXCUI_ing','TTY_ing','ingredient_name']
drug_ingredient_map = pd.concat([final[cols], pack_final[cols]], ignore_index=True).drop_duplicates()
drug_ingredient_map.to_csv('drug_ingredient_map.csv', index=False)
print('drug_ingredient_map:', drug_ingredient_map.shape)

# brand name map
bn_concepts = rxnorm_only[rxnorm_only['TTY'] == 'BN'][['RXCUI','STR']].drop_duplicates().rename(columns={'STR':'brand_name'})
bn_link     = has_ing.merge(bn_concepts, left_on='RXCUI1', right_on='RXCUI')
sbd_ing     = final[final['TTY_drug'] == 'SBD'][['RXCUI_drug','ingredient_name']].drop_duplicates()
bn_ingredient_map = (
    bn_link.merge(sbd_ing, left_on='RXCUI2', right_on='RXCUI_drug')
    [['brand_name','ingredient_name']].drop_duplicates()
    .sort_values('brand_name').reset_index(drop=True)
)
bn_ingredient_map.to_csv('bn_ingredient_map.csv', index=False)
print('bn_ingredient_map:', bn_ingredient_map.shape)

## Cell 4 — Load DDInter + match to RXCUI

In [ ]:
codes   = ['A','B','D','H','L','P','R','V']
base_url = 'https://ddinter.scbdd.com/static/media/download/ddinter_downloads_code_{}.csv'
dfs     = [pd.read_csv(base_url.format(c)) for c in codes]
ddinter = pd.concat(dfs, ignore_index=True)

ddinter['id_min'] = ddinter[['DDInterID_A','DDInterID_B']].min(axis=1)
ddinter['id_max'] = ddinter[['DDInterID_A','DDInterID_B']].max(axis=1)
positive_pairs    = ddinter[['id_min','id_max','Level']].drop_duplicates().reset_index(drop=True)

id_name = pd.concat([
    ddinter[['DDInterID_A','Drug_A']].rename(columns={'DDInterID_A':'id','Drug_A':'name'}),
    ddinter[['DDInterID_B','Drug_B']].rename(columns={'DDInterID_B':'id','Drug_B':'name'})
], ignore_index=True).drop_duplicates()
ddinter_drugs = id_name.copy()
ddinter_drugs['name_lower'] = ddinter_drugs['name'].str.lower().str.strip()

ingredient_rxcuis = set(ingredients['RXCUI'])

# --- pass 1: direct name match ---
matched = ddinter_drugs.merge(ingredients[['RXCUI','name_lower']], on='name_lower', how='left')
direct  = matched[matched['RXCUI'].notna()][['id','RXCUI']]

# --- pass 2: wider synonym match (all SABs) ---
all_alt_names = rxnconso[
    rxnconso['RXCUI'].isin(ingredient_rxcuis) &
    rxnconso['TTY'].isin(['SY','PSN','TMSY','SU','IN','FSY','PT','FN','GN'])
][['RXCUI','STR']].drop_duplicates()
all_alt_names['name_lower'] = all_alt_names['STR'].str.lower().str.strip()
alt_names = all_alt_names  # used in predict_interaction later

still_unmatched = matched[matched['RXCUI'].isna()][['id','name_lower']]
recovered = still_unmatched.merge(
    all_alt_names[['RXCUI','name_lower']], on='name_lower'
).drop_duplicates(subset='id')

ddinter_id_to_rxcui = pd.concat(
    [direct, recovered[['id','RXCUI']]], ignore_index=True
).drop_duplicates(subset='id')

print(f'Matched: {ddinter_id_to_rxcui.shape[0]} / {ddinter_drugs["id"].nunique()} DDInter drugs')

# translate pairs to RXCUI
id_map = ddinter_id_to_rxcui.set_index('id')['RXCUI'].to_dict()
positive_pairs['rxcui_min'] = positive_pairs['id_min'].map(id_map)
positive_pairs['rxcui_max'] = positive_pairs['id_max'].map(id_map)
mapped_positive = positive_pairs.dropna(subset=['rxcui_min','rxcui_max']).copy()
mapped_positive['pair_a'] = mapped_positive[['rxcui_min','rxcui_max']].min(axis=1)
mapped_positive['pair_b'] = mapped_positive[['rxcui_min','rxcui_max']].max(axis=1)
mapped_positive = mapped_positive.drop_duplicates(subset=['pair_a','pair_b'])
print('Usable interaction pairs:', mapped_positive.shape[0])

## Cell 5 — Negative sampling + Train/Test split

In [ ]:
rng = np.random.default_rng(42)
positive_set    = set(zip(mapped_positive['pair_a'], mapped_positive['pair_b']))
unique_rxcuis   = pd.concat([mapped_positive['pair_a'], mapped_positive['pair_b']]).unique()
n_unique        = len(unique_rxcuis)
n_needed        = len(positive_set)

negative_pairs = set()
while len(negative_pairs) < n_needed:
    batch = (n_needed - len(negative_pairs)) * 3
    a = unique_rxcuis[rng.integers(0, n_unique, size=batch)]
    b = unique_rxcuis[rng.integers(0, n_unique, size=batch)]
    for x, y in zip(a, b):
        if x == y: continue
        pair = (x,y) if x < y else (y,x)
        if pair in positive_set or pair in negative_pairs: continue
        negative_pairs.add(pair)
        if len(negative_pairs) >= n_needed: break

negative_df = pd.DataFrame(list(negative_pairs), columns=['pair_a','pair_b'])
negative_df['label'] = 0
positive_df = mapped_positive[['pair_a','pair_b']].copy()
positive_df['label'] = 1
dataset = pd.concat([positive_df, negative_df], ignore_index=True)

train_df, test_df = train_test_split(
    dataset, test_size=0.2, stratify=dataset['label'], random_state=42
)
print('Dataset:', dataset.shape)
print('Train:', train_df.shape, '| Test:', test_df.shape)

## Cell 6 — Fetch PubChem properties + SMILES (runs once, ~15 min total)

In [ ]:
rxcui_in_dataset = set(dataset['pair_a']) | set(dataset['pair_b'])
rxcui_name_map   = (
    ingredients[ingredients['RXCUI'].isin(rxcui_in_dataset)]
    [['RXCUI','ingredient_name']].drop_duplicates(subset='RXCUI').reset_index(drop=True)
)

PROPS      = ['MolecularWeight','XLogP','TPSA','HBondDonorCount',
              'HBondAcceptorCount','RotatableBondCount','HeavyAtomCount']
PROP_URL   = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name'
              '/{name}/property/' + ','.join(PROPS) + '/JSON')
SMILES_URL = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name'
              '/{name}/property/IsomericSMILES/JSON')

prop_results, smiles_results, failed = [], [], []

for i, row in rxcui_name_map.iterrows():
    name = requests.utils.quote(row['ingredient_name'])
    # properties
    try:
        r = requests.get(PROP_URL.format(name=name), timeout=10)
        if r.status_code == 200:
            p = r.json()['PropertyTable']['Properties'][0]
            p['RXCUI'] = row['RXCUI']
            prop_results.append(p)
    except: pass
    # SMILES
    try:
        r = requests.get(SMILES_URL.format(name=name), timeout=10)
        if r.status_code == 200:
            smi = r.json()['PropertyTable']['Properties'][0]['SMILES']
            smiles_results.append({'RXCUI': row['RXCUI'], 'SMILES': smi})
        else:
            failed.append(row['ingredient_name'])
    except:
        failed.append(row['ingredient_name'])

    if (i+1) % 200 == 0:
        print(f'  {i+1}/{len(rxcui_name_map)} — props: {len(prop_results)}, smiles: {len(smiles_results)}')
    time.sleep(0.22)

mol_props = pd.DataFrame(prop_results)
mol_props[PROPS] = mol_props[PROPS].apply(pd.to_numeric, errors='coerce')
mol_props[PROPS] = mol_props[PROPS].apply(lambda c: c.fillna(c.median()))
global_median = mol_props[PROPS].median()

all_rxcuis     = pd.DataFrame({'RXCUI': list(rxcui_in_dataset)})
mol_props_full = all_rxcuis.merge(mol_props[['RXCUI']+PROPS], on='RXCUI', how='left')
for col in PROPS:
    mol_props_full[col] = mol_props_full[col].fillna(global_median[col])

smiles_df = pd.DataFrame(smiles_results)
print(f'Properties: {len(mol_props)}, SMILES: {len(smiles_df)}, failed: {len(failed)}')

## Cell 7 — Generate Morgan fingerprints

In [ ]:
nBits = 1024
fingerprints = {}
for _, row in smiles_df.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol is None: continue
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=nBits)
    arr = np.zeros(nBits, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fingerprints[row['RXCUI']] = arr

fp_rxcuis    = list(fingerprints.keys())
fp_matrix    = np.stack([fingerprints[r] for r in fp_rxcuis])
rxcui_to_idx = {r: i for i, r in enumerate(fp_rxcuis)}
print(f'Fingerprint matrix: {fp_matrix.shape}')

## Cell 8 — Feature extraction helpers

In [ ]:
PROPS = ['MolecularWeight','XLogP','TPSA','HBondDonorCount',
         'HBondAcceptorCount','RotatableBondCount','HeavyAtomCount']
mol_feat_cols = [f'{fn}_{p}' for fn in ['diff','sum'] for p in PROPS]

def extract_graph_features(df, G):
    rows = []
    for _, row in df.iterrows():
        a, b = row['pair_a'], row['pair_b']
        da = G.degree(a) if G.has_node(a) else 0
        db = G.degree(b) if G.has_node(b) else 0
        if G.has_node(a) and G.has_node(b):
            na, nb = set(G.neighbors(a)), set(G.neighbors(b))
            common  = len(na & nb)
            union   = len(na | nb)
            jaccard = common/union if union else 0
            adamic  = sum(1/np.log(G.degree(n)) for n in (na&nb) if G.degree(n)>1)
        else:
            common = jaccard = adamic = 0
        rows.append([da, db, common, jaccard, adamic])
    return pd.DataFrame(rows,
        columns=['deg_a','deg_b','common_neighbors','jaccard','adamic_adar'],
        index=df.index)

def add_mol_features(df, mol_props_full, props):
    m = mol_props_full.set_index('RXCUI')[props]
    res = df.copy()
    for p in props:
        a_vals = res['pair_a'].map(m[p])
        b_vals = res['pair_b'].map(m[p])
        res[f'diff_{p}'] = (a_vals - b_vals).abs()
        res[f'sum_{p}']  =  a_vals + b_vals
    return res

def add_fingerprint_features(df, fp_matrix, rxcui_to_idx):
    has_a = df['pair_a'].isin(rxcui_to_idx)
    has_b = df['pair_b'].isin(rxcui_to_idx)
    both  = has_a & has_b
    result = df.copy()
    result['tanimoto'] = 0.0
    result['has_fingerprint'] = both.astype(int)
    if both.sum() > 0:
        idx_a = df.loc[both, 'pair_a'].map(rxcui_to_idx).values
        idx_b = df.loc[both, 'pair_b'].map(rxcui_to_idx).values
        A = fp_matrix[idx_a]; B = fp_matrix[idx_b]
        inter = np.logical_and(A,B).sum(axis=1)
        union = np.logical_or(A,B).sum(axis=1)
        result.loc[both, 'tanimoto'] = np.where(union>0, inter/union, 0.0)
    return result

print('Feature helpers defined')

## Cell 9 — Build interaction graph + extract all features

In [ ]:
train_positive = train_df[train_df['label'] == 1]
G = nx.Graph()
G.add_edges_from(zip(train_positive['pair_a'], train_positive['pair_b']))
print(f'Graph — nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}')

print('Extracting graph features...')
X_train_g = extract_graph_features(train_df, G)
X_test_g  = extract_graph_features(test_df,  G)

print('Adding molecular features...')
train_mol = add_mol_features(train_df, mol_props_full, PROPS)
test_mol  = add_mol_features(test_df,  mol_props_full, PROPS)

print('Adding fingerprint features...')
train_fp  = add_fingerprint_features(train_df, fp_matrix, rxcui_to_idx)
test_fp   = add_fingerprint_features(test_df,  fp_matrix, rxcui_to_idx)

X_train = pd.concat([
    X_train_g.reset_index(drop=True),
    train_mol[mol_feat_cols].reset_index(drop=True),
    train_fp[['tanimoto','has_fingerprint']].reset_index(drop=True)
], axis=1)

X_test = pd.concat([
    X_test_g.reset_index(drop=True),
    test_mol[mol_feat_cols].reset_index(drop=True),
    test_fp[['tanimoto','has_fingerprint']].reset_index(drop=True)
], axis=1)

y_train = train_df['label'].reset_index(drop=True)
y_test  = test_df['label'].reset_index(drop=True)
feature_columns = list(X_train.columns)
print('Feature matrix:', X_train.shape)

## Cell 10 — Train Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Interaction','Interaction']))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 4))

feat_imp = pd.Series(rf.feature_importances_, index=feature_columns).sort_values(ascending=False)
print('\nTop 10 features:')
print(feat_imp.head(10))

## Cell 11 — Save model + support data

In [ ]:
joblib.dump(rf, 'ddi_final_model_v2.pkl')
joblib.dump({'feature_columns': feature_columns,
             'roc_auc': round(roc_auc_score(y_test, y_proba), 4)},
            'ddi_model_metadata_v2.pkl')
joblib.dump({
    'ingredients':      ingredients,
    'bn_ingredient_map':bn_ingredient_map,
    'alt_names':        alt_names,
    'mol_props_full':   mol_props_full,
    'fp_matrix':        fp_matrix,
    'rxcui_to_idx':     rxcui_to_idx,
    'positive_set':     positive_set,
}, 'ddi_support_data_v2.pkl')
print('All files saved.')

### B — Severity v2 build (binary classifiers + overrides)
Builds the two binary severity detectors and the override list.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import numpy as np

# rebuild severity dataset
severity_pairs = mapped_positive[['pair_a','pair_b','Level']].copy()
severity_known = severity_pairs[severity_pairs['Level'] != 'Unknown'].copy()

# build features
sev_graph_f = extract_graph_features(severity_known, G)
sev_mol     = add_mol_features(severity_known, mol_props_full, PROPS)
sev_fp      = add_fingerprint_features(severity_known, fp_matrix, rxcui_to_idx)

X_sev = pd.concat([
    sev_graph_f.reset_index(drop=True),
    sev_mol[mol_feat_cols].reset_index(drop=True),
    sev_fp[['tanimoto','has_fingerprint']].reset_index(drop=True)
], axis=1)
y_sev = severity_known['Level'].reset_index(drop=True)

X_sev_train, X_sev_test, y_sev_train, y_sev_test = train_test_split(
    X_sev, y_sev, test_size=0.2, stratify=y_sev, random_state=42
)

# fill NaN with training medians
train_medians      = X_sev_train.median()
X_sev_train_clean  = X_sev_train.fillna(train_medians)
X_sev_test_clean   = X_sev_test.fillna(train_medians)

# binary labels
y_is_major = (y_sev_train == 'Major').astype(int)
y_is_minor = (y_sev_train == 'Minor').astype(int)

rf_major = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=42,
    class_weight={0: 1, 1: 4}
)
rf_major.fit(X_sev_train_clean, y_is_major)

rf_minor = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=42,
    class_weight={0: 1, 1: 10}
)
rf_minor.fit(X_sev_train_clean, y_is_minor)

MAJOR_THRESHOLD = 0.35
MINOR_THRESHOLD = 0.15

major_proba = rf_major.predict_proba(X_sev_test_clean)[:, 1]
minor_proba = rf_minor.predict_proba(X_sev_test_clean)[:, 1]

y_final = []
for mp, mn in zip(major_proba, minor_proba):
    if mp >= MAJOR_THRESHOLD:   y_final.append('Major')
    elif mn >= MINOR_THRESHOLD: y_final.append('Minor')
    else:                       y_final.append('Moderate')

print(classification_report(y_sev_test, y_final,
      target_names=['Major','Minor','Moderate']))

print('Major recall:', round(sum(1 for t,p in zip(y_sev_test, y_final) if t=='Major' and p=='Major') / sum(1 for t in y_sev_test if t=='Major'), 2))
print('Minor recall:', round(sum(1 for t,p in zip(y_sev_test, y_final) if t=='Minor' and p=='Minor') / sum(1 for t in y_sev_test if t=='Minor'), 2))


In [ ]:
# hard-coded override list for critical known Major pairs
def build_override_list():
    pairs = [
        ('Warfarin',     'Aspirin'),
        ('Warfarin',     'Ibuprofen'),
        ('Warfarin',     'Naproxen'),
        ('Warfarin',     'Diclofenac'),
        ('Warfarin',     'Fluconazole'),
        ('Warfarin',     'Amiodarone'),
        ('Warfarin',     'Metronidazole'),
        ('Warfarin',     'Ciprofloxacin'),
        ('Methotrexate', 'Aspirin'),
        ('Methotrexate', 'Ibuprofen'),
        ('Methotrexate', 'Naproxen'),
        ('Lithium',      'Ibuprofen'),
        ('Lithium',      'Naproxen'),
        ('Digoxin',      'Amiodarone'),
        ('Digoxin',      'Clarithromycin'),
        ('Phenytoin',    'Fluconazole'),
        ('Phenytoin',    'Amiodarone'),
        ('Simvastatin',  'Amiodarone'),
        ('Simvastatin',  'Clarithromycin'),
        ('Simvastatin',  'Fluconazole'),
    ]
    overrides = {}
    for a, b in pairs:
        ra, _ = resolve_rxcui(a)
        rb, _ = resolve_rxcui(b)
        if ra and rb:
            overrides[(min(ra,rb), max(ra,rb))] = 'Major'
    print(f'Override list: {len(overrides)} pairs')
    return overrides

SEVERITY_OVERRIDES = build_override_list()

def predict_severity(X, rxcui_a, rxcui_b):
    key = (min(rxcui_a, rxcui_b), max(rxcui_a, rxcui_b))
    if key in SEVERITY_OVERRIDES:
        return SEVERITY_OVERRIDES[key], 'OVERRIDE', {
            'major_probability': None, 'minor_probability': None,
            'note': 'Severity from verified clinical override list'
        }
    X_clean  = X.fillna(train_medians)
    major_p  = rf_major.predict_proba(X_clean)[0, 1]
    minor_p  = rf_minor.predict_proba(X_clean)[0, 1]
    in_gray  = (0.25 <= major_p < 0.35)
    if major_p >= MAJOR_THRESHOLD:
        severity, top_prob, uncertain = 'Major',    major_p,       False
    elif minor_p >= MINOR_THRESHOLD:
        severity, top_prob, uncertain = 'Minor',    minor_p,       False
    else:
        severity, top_prob, uncertain = 'Moderate', 1 - major_p,   in_gray
    if uncertain:
        sev_conf = 'UNCERTAIN'
    elif top_prob >= 0.70:
        sev_conf = 'HIGH'
    elif top_prob >= 0.55:
        sev_conf = 'MEDIUM'
    else:
        sev_conf = 'LOW'
    return severity, sev_conf, {
        'major_probability': round(float(major_p), 3),
        'minor_probability': round(float(minor_p), 3),
    }

print('predict_severity() defined with overrides and gray zone flag.')


In [ ]:
joblib.dump(rf_major, 'ddi_severity_major.pkl')
joblib.dump(rf_minor, 'ddi_severity_minor.pkl')
joblib.dump({
    'major_threshold': MAJOR_THRESHOLD,
    'minor_threshold': MINOR_THRESHOLD,
    'train_medians':   train_medians.to_dict(),
    'overrides':       SEVERITY_OVERRIDES,
}, 'ddi_severity_config.pkl')
print('Saved: ddi_severity_major.pkl')
print('Saved: ddi_severity_minor.pkl')
print('Saved: ddi_severity_config.pkl')
print(f'Override list: {len(SEVERITY_OVERRIDES)} critical pairs')
print(f'Major threshold: {MAJOR_THRESHOLD}')
print(f'Minor threshold: {MINOR_THRESHOLD}')


### B — Expand overrides to all DDInter Major pairs

In [ ]:
import pandas as pd

# load saved DDInter tables (no internet needed)
positive_pairs    = pd.read_csv('ddinter_positive_pairs.csv')
ddinter_drugs_df  = pd.read_csv('ddinter_drugs.csv')
ddinter_id_to_rxcui = pd.read_csv('ddinter_id_to_rxcui.csv')

# translate pairs to RXCUI
id_map = ddinter_id_to_rxcui.set_index('id')['RXCUI'].to_dict()
positive_pairs['rxcui_min'] = positive_pairs['id_min'].map(id_map)
positive_pairs['rxcui_max'] = positive_pairs['id_max'].map(id_map)
mapped_positive = positive_pairs.dropna(subset=['rxcui_min','rxcui_max']).copy()
mapped_positive['pair_a'] = mapped_positive[['rxcui_min','rxcui_max']].min(axis=1)
mapped_positive['pair_b'] = mapped_positive[['rxcui_min','rxcui_max']].max(axis=1)
mapped_positive = mapped_positive.drop_duplicates(subset=['pair_a','pair_b'])

# build expanded override list
major_pairs = mapped_positive[mapped_positive['Level'] == 'Major'].copy()
major_pairs['key_min'] = major_pairs[['pair_a','pair_b']].min(axis=1)
major_pairs['key_max'] = major_pairs[['pair_a','pair_b']].max(axis=1)
major_pairs = major_pairs.drop_duplicates(subset=['key_min','key_max'])

SEVERITY_OVERRIDES = {
    (row['key_min'], row['key_max']): 'Major'
    for _, row in major_pairs.iterrows()
}
print(f'Override list: {len(SEVERITY_OVERRIDES):,} Major pairs')

# save updated config
joblib.dump({
    'major_threshold': MAJOR_THRESHOLD,
    'minor_threshold': MINOR_THRESHOLD,
    'train_medians':   train_medians.to_dict(),
    'overrides':       SEVERITY_OVERRIDES,
}, 'ddi_severity_config.pkl')
print('Saved: ddi_severity_config.pkl')


### B — Pharmacophore allergy classes (13 classes)

In [ ]:
import re, joblib

ALLERGY_CLASS_KEYWORDS = {
    'beta_lactam': ['penicillin','amoxicillin','ampicillin','oxacillin','nafcillin',
        'cloxacillin','dicloxacillin','flucloxacillin','floxacillin','piperacillin',
        'ticarcillin','carbenicillin','mezlocillin','azlocillin','methicillin',
        'bacampicillin','pivampicillin','cephalexin','cefazolin','cefuroxime',
        'ceftriaxone','cefotaxime','cefixime','cefdinir','cefpodoxime','cefepime',
        'ceftazidime','cefaclor','cefadroxil','cefprozil','cefoxitin','cefotetan',
        'loracarbef','imipenem','meropenem','ertapenem','doripenem','aztreonam',
        'clemizolpenicillin','talampicillin'],
    'sulfonamide': ['sulfamethoxazole','sulfadiazine','sulfasalazine','sulfisoxazole',
        'sulfacetamide','sulfadoxine','sulfamethizole','sulfanilamide',
        'sulfapyridine','sulfathiazole','dapsone'],
    'nsaid': ['ibuprofen','naproxen','diclofenac','indomethacin','ketoprofen',
        'piroxicam','meloxicam','celecoxib','rofecoxib','valdecoxib','etoricoxib',
        'flurbiprofen','fenoprofen','oxaprozin','tolmetin','sulindac','etodolac',
        'nabumetone','mefenamic acid','meclofenamate','phenylbutazone','ketorolac',
        'diflunisal','salsalate','aspirin','acetylsalicylic'],
    'opioid': ['codeine','morphine','hydrocodone','oxycodone','hydromorphone',
        'oxymorphone','fentanyl','methadone','meperidine','tramadol','buprenorphine',
        'nalbuphine','butorphanol','pentazocine','tapentadol','dihydrocodeine',
        'levorphanol','benzhydrocodone','oliceridine'],
    'fluoroquinolone': ['ciprofloxacin','levofloxacin','moxifloxacin','ofloxacin',
        'norfloxacin','gemifloxacin','delafloxacin','sparfloxacin','gatifloxacin',
        'lomefloxacin','enrofloxacin','marbofloxacin'],
    'tetracycline': ['tetracycline','doxycycline','minocycline','demeclocycline',
        'tigecycline','oxytetracycline','chlortetracycline','eravacycline',
        'omadacycline','sarecycline'],
    'macrolide': ['erythromycin','azithromycin','clarithromycin','roxithromycin',
        'telithromycin','fidaxomicin','spiramycin','josamycin'],
    'statin': ['atorvastatin','simvastatin','rosuvastatin','pravastatin',
        'lovastatin','fluvastatin','pitavastatin','cerivastatin'],
    'ace_inhibitor': ['lisinopril','enalapril','ramipril','captopril','benazepril',
        'fosinopril','quinapril','perindopril','trandolapril','moexipril',
        'cilazapril','spirapril','zofenopril'],
    'arb': ['losartan','valsartan','irbesartan','candesartan','olmesartan',
        'telmisartan','eprosartan','azilsartan'],
    'para_aminophenol': ['acetaminophen','paracetamol','phenacetin','propacetamol'],
    'lincosamide': ['clindamycin','lincomycin','pirlimycin'],
    'beta_blocker': ['bisoprolol','metoprolol','atenolol','propranolol','carvedilol',
        'nebivolol','labetalol','nadolol','timolol','acebutolol',
        'pindolol','oxprenolol','sotalol','esmolol','betaxolol'],
}

PHARMACOPHORE_CLASSES = {}
for class_name, keywords in ALLERGY_CLASS_KEYWORDS.items():
    rxcuis = set()
    for kw in keywords:
        hits = ingredients[ingredients['ingredient_name'].str.contains(kw, case=False, na=False)]['RXCUI'].tolist()
        rxcuis.update(hits)
    PHARMACOPHORE_CLASSES[class_name] = rxcuis
    print(f'{class_name:20}: {len(rxcuis):4} RXCUIs')

# regional synonyms — 54 brand names not in RxNorm
REGIONAL_SYNONYMS = {
    # Analgesics
    'nurofen':'5640','nurofen plus':'5640','brustan':'5640','profinal':'5640','rivo':'5640',
    'ketofan':'6142','novalgin':'3523','dolalgial':'3523','novalgine':'3523',
    'adol':'161','fevadol':'161','doliprane':'161','efferalgan':'161','dafalgan':'161',
    'biogesic':'161','dolo':'161','tylenol extra':'161','panadeine':'161',
    'syndol':'161','saridon':'161','disprin':'1191','aspro':'1191','thomapyrin':'1191',
    'combiflam':'5640',
    # Antibiotics
    'ospamox':'723','clamoxyl':'723','azithrin':'18631','vinzam':'18631',
    'ciprobay':'2551','cefspan':'25033','dalacin':'2582',
    'klacid':'21212','klaricid':'21212','doxine':'3640',
    # Cardiovascular
    'marevan':'11289','iscover':'32968','istin':'17767',
    'concor':'19484','emconcor':'19484','tritace':'35296','aprovel':'83818',
    # Diabetes
    'diaformin':'6809','glicomet':'6809','glucocil':'6809','diamicron':'4816',
    # Lipid lowering
    'torvast':'83367',
    # Respiratory/Allergy
    'seretide':'36117','flixotide':'41126','telfast':'87636',
    'aerius':'275635','claritine':'28889',
    # CNS
    'lexotanil':'1749','trittico':'10737',
    # Other
    'solucortef':'5492',
}

joblib.dump({
    'tanimoto_threshold':    0.16,
    'atc_min_tanimoto':      0.12,
    'pharmacophore_classes': {k: list(v) for k, v in PHARMACOPHORE_CLASSES.items()},
    'allergy_keywords':      ALLERGY_CLASS_KEYWORDS,
}, 'ddi_allergy_config.pkl')

support = joblib.load('ddi_support_data_v2.pkl')
support['regional_synonyms'] = REGIONAL_SYNONYMS
joblib.dump(support, 'ddi_support_data_v2.pkl')
print(f'Saved. Classes: {len(PHARMACOPHORE_CLASSES)}, Regional synonyms: {len(REGIONAL_SYNONYMS)}')


### B — Pregnancy database (1,712 drugs from openFDA)

In [ ]:
import re, joblib, json

# load pregnancy data
preg_data           = joblib.load('ddi_pregnancy_data.pkl')
PREGNANCY_OVERRIDES = preg_data['overrides']
pregnancy_db        = preg_data['pregnancy_db']
print(f'Pregnancy DB: {len(pregnancy_db)} drugs, Overrides: {len(PREGNANCY_OVERRIDES)}')

PREGNANCY_RULES = [
    (r'contraindicated.{0,80}pregnan|pregnan.{0,80}contraindicated|category\s*x|do not use.{0,50}pregnan',
     'X', 'Contraindicated in pregnancy — risk of fetal harm'),
    (r'teratogen|embryotoxic|fetotoxic|fetal.{0,30}malform|birth defect',
     'D', 'Evidence of fetal risk — use only if benefits outweigh risks'),
    (r'fetal harm|fetal risk|adverse.{0,50}fetal|category\s*d|neonatal.{0,50}death',
     'D', 'Evidence of fetal risk — use only if benefits outweigh risks'),
    (r'20 weeks|after 20|third trimester|last 3 months|oligohydramnios|problems in the unborn',
     'D*', 'Avoid in late pregnancy (after 20 weeks) — risk of fetal complications'),
    (r'no adequate.{0,50}well.controlled|no adequate.{0,50}studies.{0,50}pregnant|cross.{0,30}placenta|category\s*c|benefit.{0,50}outweigh',
     'C', 'Use with caution — no adequate human studies'),
    (r'category\s*b|no evidence.{0,50}harm|animal.{0,50}no.{0,50}risk|not reported a clear',
     'B', 'Relatively safe — no evidence of harm in animal studies'),
    (r'not sufficient to inform|insufficient data|limited.{0,50}data|no information.{0,30}pregnan',
     'N/A', 'Insufficient data — consult physician'),
    (r'category\s*a|no risk.{0,50}controlled.{0,50}studies',
     'A', 'Safe — controlled human studies show no risk'),
]

def fetch_pregnancy_openfda(drug_name):
    import requests
    url    = 'https://api.fda.gov/drug/label.json'
    params = {'search': f'openfda.generic_name:"{drug_name}"', 'limit': 1}
    resp   = requests.get(url, params=params, timeout=10)
    if resp.status_code != 200: return {'error': f'API failed: {resp.status_code}'}
    data   = resp.json()
    if 'results' not in data or len(data['results']) == 0: return {'error': 'No label found'}
    label  = data['results'][0]
    fields = ['pregnancy','pregnancy_or_breast_feeding','teratogenic_effects',
              'nonteratogenic_effects','use_in_specific_populations','nursing_mothers']
    found  = {}
    for field in fields:
        if field in label:
            text = ' '.join(label[field]) if isinstance(label[field], list) else str(label[field])
            found[field] = text[:300]
    return {'drug': drug_name, 'source': 'openFDA', 'data': found}

def extract_pregnancy_category(text, drug_name=None):
    if drug_name and drug_name.lower() in PREGNANCY_OVERRIDES:
        cat, warn = PREGNANCY_OVERRIDES[drug_name.lower()]
        return cat, warn, 'clinical_override'
    text_lower = text.lower()
    for pattern, category, warning in PREGNANCY_RULES:
        if re.search(pattern, text_lower):
            return category, warning, 'text_extraction'
    return 'Unknown', 'No pregnancy data found — consult physician', 'not_found'

def get_pregnancy_info(drug_name):
    rxcui, _ = resolve_rxcui(drug_name)
    ingredient = drug_name
    if rxcui:
        ing = ingredients[ingredients['RXCUI'] == rxcui]['ingredient_name'].values
        if len(ing) > 0: ingredient = ing[0]
    pllr_note = None
    if ingredient in pregnancy_db:
        data = pregnancy_db[ingredient]
        cat  = data['category']; warn = data['warning']; src = data['source']
        if src != 'clinical_override':
            raw      = fetch_pregnancy_openfda(ingredient)
            all_text = ' '.join(raw.get('data', {}).values()) if 'data' in raw else ''
            match    = re.search(r'risk summary\s+(.{50,250}?)(?:\.|in animal|because|treatment)',
                                  all_text, re.IGNORECASE)
            if match: pllr_note = match.group(1).strip()
        return {'drug': drug_name, 'ingredient': ingredient,
                'category': cat, 'warning': warn, 'source': src, 'pllr_note': pllr_note}
    raw      = fetch_pregnancy_openfda(ingredient)
    all_text = ' '.join(raw.get('data', {}).values()) if 'data' in raw else ''
    cat, warn, src = extract_pregnancy_category(all_text, ingredient)
    match = re.search(r'risk summary\s+(.{50,250}?)(?:\.|in animal|because|treatment)',
                       all_text, re.IGNORECASE)
    if match: pllr_note = match.group(1).strip()
    return {'drug': drug_name, 'ingredient': ingredient,
            'category': cat, 'warning': warn, 'source': src, 'pllr_note': pllr_note}

def check_pregnancy(name_a, name_b=None):
    result_a = get_pregnancy_info(name_a)
    if name_b is None:
        return {'drug': name_a, 'ingredient': result_a['ingredient'],
                'category': result_a['category'], 'warning': result_a['warning'],
                'pllr_note': result_a.get('pllr_note'), 'source': result_a['source'],
                'note': 'Always consult a physician before taking any medication during pregnancy.'}
    result_b  = get_pregnancy_info(name_b)
    risk_rank = {'X':5,'D':4,'D*':3,'C':2,'B':1,'A':0,'N/A':1,'Unknown':1}
    highest   = result_a['category'] if risk_rank.get(result_a['category'],1) >= risk_rank.get(result_b['category'],1) else result_b['category']
    advice    = {'X':'⛔ AVOID — one or both drugs are contraindicated in pregnancy',
                 'D':'🔴 HIGH RISK — use only if no safe alternative exists',
                 'D*':'🟠 AVOID IN LATE PREGNANCY — especially after 20 weeks',
                 'C':'🟡 USE WITH CAUTION — consult physician before use',
                 'B':'🟢 RELATIVELY SAFE — but always confirm with physician',
                 'A':'✅ SAFE — generally considered safe in pregnancy',
                 }.get(highest, '⚪ UNKNOWN — consult physician')
    return {
        'drug_a': {'name': name_a, 'ingredient': result_a['ingredient'],
                   'category': result_a['category'], 'warning': result_a['warning'],
                   'pllr_note': result_a.get('pllr_note')},
        'drug_b': {'name': name_b, 'ingredient': result_b['ingredient'],
                   'category': result_b['category'], 'warning': result_b['warning'],
                   'pllr_note': result_b.get('pllr_note')},
        'overall_risk': highest, 'overall_advice': advice,
        'note': 'Always consult a physician before taking any medication during pregnancy.'
    }

# test
print(json.dumps(check_pregnancy('apixaban'), indent=2))
print(json.dumps(check_pregnancy('Warfarin', 'Aspirin'), indent=2))


### B — Critical new-drug overrides (post-2010 drugs)

In [ ]:
import joblib

# all manually verified critical overrides for newer drugs not in DDInter
new_critical_overrides = {
    # dabigatran (P-gp inhibitors)
    ('dabigatran','clarithromycin'): 'Major',
    ('dabigatran','ketoconazole'):   'Major',
    ('dabigatran','amiodarone'):     'Major',
    ('dabigatran','aspirin'):        'Major',
    # apixaban / rivaroxaban (CYP3A4 + P-gp inhibitors)
    ('apixaban','clarithromycin'):   'Major',
    ('rivaroxaban','clarithromycin'):'Major',
    ('rivaroxaban','ketoconazole'):  'Major',
    ('rivaroxaban','amiodarone'):    'Major',
    # sacubitril (angioedema with ACE inhibitors)
    ('sacubitril','lisinopril'):     'Major',
    ('sacubitril','enalapril'):      'Major',
    ('sacubitril','ramipril'):       'Major',
    ('sacubitril','captopril'):      'Major',
    # linezolid (serotonin syndrome)
    ('linezolid','fluoxetine'):      'Major',
    ('linezolid','sertraline'):      'Major',
    ('linezolid','tramadol'):        'Major',
    ('linezolid','citalopram'):      'Major',
    ('linezolid','escitalopram'):    'Major',
    ('linezolid','venlafaxine'):     'Major',
    # daptomycin (rhabdomyolysis with statins)
    ('daptomycin','simvastatin'):    'Major',
    ('daptomycin','atorvastatin'):   'Major',
    ('daptomycin','rosuvastatin'):   'Major',
    # sofosbuvir (fatal bradycardia — FDA 2015 safety communication)
    ('sofosbuvir','amiodarone'):     'Major',
    # ledipasvir + statins (Harvoni label — significant statin level increase)
    ('ledipasvir','atorvastatin'):   'Major',
    ('ledipasvir','simvastatin'):    'Major',
    ('ledipasvir','rosuvastatin'):   'Major',
    # tedizolid + SSRIs/SNRIs (MAO inhibition class effect)
    ('tedizolid','fluoxetine'):      'Major',
    ('tedizolid','sertraline'):      'Major',
    ('tedizolid','tramadol'):        'Major',
    ('tedizolid','venlafaxine'):     'Major',
    ('tedizolid','escitalopram'):    'Major',
}

# reload and update
sev_config = joblib.load('ddi_severity_config.pkl')
SEVERITY_OVERRIDES = sev_config['overrides']
added = 0
for (name_a, name_b), severity in new_critical_overrides.items():
    ra, _ = resolve_rxcui(name_a)
    rb, _ = resolve_rxcui(name_b)
    if ra and rb:
        key = (min(ra,rb), max(ra,rb))
        SEVERITY_OVERRIDES[key] = severity
        added += 1
sev_config['overrides'] = SEVERITY_OVERRIDES
joblib.dump(sev_config, 'ddi_severity_config.pkl')
print(f'Added: {added} pairs | Total overrides: {len(SEVERITY_OVERRIDES):,}')
print('Saved: ddi_severity_config.pkl')

# verify
for a, b in [('sofosbuvir','amiodarone'),('ledipasvir','atorvastatin'),
             ('tedizolid','fluoxetine'),('dabigatran','clarithromycin')]:
    r = predict_interaction(a, b)
    print(f'  {a} + {b}: {r["severity"]} ({r["severity_confidence"]})')


In [3]:
from ddi_engine import DDIEngine

engine = DDIEngine(data_dir="./")
print(engine.predict_interaction("Warfarin", "Aspirin"))

{'drug_a': 'Warfarin', 'resolved_as_a': 'ingredient', 'drug_b': 'Aspirin', 'resolved_as_b': 'ingredient', 'interaction_probability': 1.0, 'prediction': 'Interaction likely', 'severity': 'Major', 'severity_confidence': 'OVERRIDE', 'severity_confidence_explanation': 'Severity verified from DDInter clinical data — highest reliability', 'severity_probabilities': {'major_probability': None, 'minor_probability': None, 'note': 'Severity from verified clinical override list'}, 'confidence': 'NORMAL', 'known_in_training_data': True, 'alternatives': {'atc_class': 'N02', 'candidates': [{'name': 'opium', 'interaction_degree': 361}, {'name': 'oliceridine', 'interaction_degree': 232}, {'name': 'lasmiditan', 'interaction_degree': 132}, {'name': 'dihydrocodeine', 'interaction_degree': 112}, {'name': 'dihydroergotamine', 'interaction_degree': 95}], 'note': 'Same ATC class as Aspirin, no recorded interaction with Warfarin. Always verify with a pharmacist.'}}
